# YOLO tile 디텍터: 증강 + train/val/test 7:2:1 분리 + YOLOv8s/YOLO26n 학습·비교

`train_yolo_tile_only.ipynb`는 원본 583(+8)장을 그대로 복사만 해서 썼고 `val`이 아예 없었습니다.
이 노트북은 그 문제를 다음 두 가지로 보강합니다.

1. **증강**: `rummikub_data_3`의 원본 이미지들에 회전(0/90/180/270) x 밝기(-25/+25) = 장당 8배
   증강을 적용합니다 (`data_argu`를 만들 때와 같은 방식). 이번엔 라벨을 전부 class_id `0`(`tile`)
   1클래스로 만듭니다.
2. **7:2:1 train/val/test 분리**: 원본 이미지 단위(그룹)로 섞어서 train:test:val = 7:2:1로
   나눕니다 — 같은 원본의 8개 증강본이 서로 다른 split에 흩어지면 leakage가 생기므로 반드시
   그룹 단위로 나눕니다. (참고: 몇 턴 전 분류기 데이터셋을 만들 때와 동일하게 순서 그대로
   `train, test, val = 7, 2, 1` 비율로 뒀습니다. 필요하면 `SPLIT_RATIOS`에서 바로 바꿀 수 있습니다.)

그 다음 이 데이터로 **YOLOv8s**와 **YOLO26n**을 각각 학습해서 가중치와 test 성능을 나란히
비교합니다 (YOLO26n은 이전에 1클래스로도 전혀 감지를 못 했던 전적이 있어 참고용/비교용으로
같이 돌리는 것이고, 기본으로 신뢰할 모델은 YOLOv8s입니다).

`rummikub_data_3` 원본은 건드리지 않고, 결과는 전부 새 폴더 `rummikub_data_3_tile_augmented`에
**실제 파일로 복사/생성**합니다 (지난번 symlink+`resolve()` 버그 재발을 막기 위해 심볼릭 링크는
쓰지 않습니다). cwd가 `.../boardgame-ai/rummikub_0730`라고 가정한 상대경로만 쓰며, 아직 실행하지
않은 상태입니다.

**주의**: 원본 591장(583 train + 8 test) x 8배 증강 = 약 4,700장의 전체 크기 원본 사진이라
디스크 용량(수십 GB 단위일 수 있음)과 생성 시간이 꽤 걸릴 수 있습니다.

In [ ]:
from __future__ import annotations

import random
import shutil
from pathlib import Path

import cv2
import numpy as np
import torch

SRC_ROOT = Path("rummikub_data_3")
OUT_ROOT = Path("rummikub_data_3_tile_augmented")
WEIGHTS_OUT = Path("weights")
WEIGHTS_OUT.mkdir(exist_ok=True)

IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png"}
ROTATIONS = (0, 90, 180, 270)
BRIGHTNESS_LEVELS = (-25, 25)

# 사용자가 말한 순서 그대로: (train, test, val) = (7, 2, 1)
SPLIT_RATIOS = {"train": 0.7, "test": 0.2, "val": 0.1}
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
assert abs(sum(SPLIT_RATIOS.values()) - 1.0) < 1e-9

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

## 1. 원본 이미지+라벨 수집 (train+test 전체를 하나의 풀로)
라벨은 여기서 바로 class_id `0`(`tile`)으로 바꿔서 읽습니다 (원본 파일은 건드리지 않음).

In [ ]:
def load_records(root: Path) -> list[dict]:
    records = []
    for split in ("train", "test"):
        image_dir = root / "images" / split
        label_dir = root / "labels" / split
        if not image_dir.is_dir():
            continue
        for image_path in sorted(image_dir.iterdir()):
            if image_path.suffix.lower() not in IMAGE_SUFFIXES:
                continue
            label_path = label_dir / f"{image_path.stem}.txt"
            if not label_path.is_file():
                continue
            lines = [
                line for line in label_path.read_text(encoding="utf-8").splitlines() if line.strip()
            ]
            if not lines:
                continue
            boxes = []
            for line in lines:
                parts = line.split()
                cx, cy, w, h = (float(v) for v in parts[1:5])
                boxes.append((cx, cy, w, h))  # class_id는 버리고 tile 1개로 통일
            records.append({"stem": image_path.stem, "image_path": image_path, "boxes": boxes})
    return records


records = load_records(SRC_ROOT)
print(f"원본 이미지 {len(records)}장, 박스 {sum(len(r['boxes']) for r in records)}개")
assert records, f"이미지를 찾지 못했습니다: {SRC_ROOT.resolve()}"

## 2. 원본 이미지 단위 그룹으로 train/test/val 7:2:1 분리
1클래스라 클래스 균형은 신경 쓸 필요가 없어서, 원본 이미지를 무작위로 섞은 뒤 비율대로
잘라 그룹(=원본 1장 + 그 8개 증강본)째로 배정합니다.

In [ ]:
def assign_groups_to_splits(stems: list[str], ratios: dict[str, float], seed: int) -> dict[str, str]:
    order = list(stems)
    random.Random(seed).shuffle(order)

    n = len(order)
    counts = {split: max(1, round(n * ratio)) for split, ratio in ratios.items()}
    diff = n - sum(counts.values())
    largest_split = max(counts, key=lambda s: counts[s])
    counts[largest_split] += diff  # 반올림 오차를 가장 큰 split(보통 train)에서 흡수

    assignment: dict[str, str] = {}
    idx = 0
    for split, count in counts.items():
        for stem in order[idx : idx + count]:
            assignment[stem] = split
        idx += count
    return assignment


assignment = assign_groups_to_splits([r["stem"] for r in records], SPLIT_RATIOS, RANDOM_SEED)
n_by_split = {split: sum(1 for s in assignment.values() if s == split) for split in SPLIT_RATIOS}
print("원본 이미지 기준 배분:", n_by_split)

## 3. 회전/밝기 증강 함수
전체 이미지를 회전시키므로 박스 좌표도 같이 회전 변환해야 합니다 (아래 공식은 `data_argu`를
만들 때 합성 이미지로 이미 검증했던 것과 동일합니다).

In [ ]:
CV2_ROTATE_CODE = {
    0: None,
    90: cv2.ROTATE_90_CLOCKWISE,
    180: cv2.ROTATE_180,
    270: cv2.ROTATE_90_COUNTERCLOCKWISE,
}


def rotate_image(image_rgb: np.ndarray, degrees: int) -> np.ndarray:
    code = CV2_ROTATE_CODE[degrees]
    return image_rgb if code is None else cv2.rotate(image_rgb, code)


def rotate_box(cx: float, cy: float, w: float, h: float, degrees: int) -> tuple[float, float, float, float]:
    if degrees == 0:
        return cx, cy, w, h
    if degrees == 90:
        return 1.0 - cy, cx, h, w
    if degrees == 180:
        return 1.0 - cx, 1.0 - cy, w, h
    if degrees == 270:
        return cy, 1.0 - cx, h, w
    raise ValueError(f"Unsupported rotation: {degrees}")


def adjust_brightness(image_rgb: np.ndarray, brightness: int) -> np.ndarray:
    if brightness == 0:
        return image_rgb
    hsv = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2HSV).astype(np.int16)
    hsv[:, :, 2] = np.clip(hsv[:, :, 2] + brightness, 0, 255)
    return cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2RGB)


def brightness_tag(value: int) -> str:
    return f"bp{value}" if value > 0 else f"bm{abs(value)}"

## 4. 증강 + 저장
이미지 하나당: 회전 4종 x 밝기 2종 = 8장을 만들어, 1번에서 정해진 split 폴더에 저장합니다.

In [ ]:
for split in SPLIT_RATIOS:
    (OUT_ROOT / "images" / split).mkdir(parents=True, exist_ok=True)
    (OUT_ROOT / "labels" / split).mkdir(parents=True, exist_ok=True)

stats = {"originals": 0, "written": 0, "unreadable": 0}

for record in records:
    split = assignment[record["stem"]]
    image_bgr = cv2.imread(str(record["image_path"]))
    if image_bgr is None:
        stats["unreadable"] += 1
        continue
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    stats["originals"] += 1

    out_image_dir = OUT_ROOT / "images" / split
    out_label_dir = OUT_ROOT / "labels" / split

    for degrees in ROTATIONS:
        rotated_image = rotate_image(image_rgb, degrees)
        rotated_boxes = [rotate_box(cx, cy, w, h, degrees) for cx, cy, w, h in record["boxes"]]
        label_lines = [f"0 {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}" for cx, cy, w, h in rotated_boxes]

        for brightness in BRIGHTNESS_LEVELS:
            final_rgb = adjust_brightness(rotated_image, brightness)
            final_bgr = cv2.cvtColor(final_rgb, cv2.COLOR_RGB2BGR)
            stem = f"{record['stem']}_r{degrees}_{brightness_tag(brightness)}"
            cv2.imwrite(str(out_image_dir / f"{stem}.jpg"), final_bgr)
            (out_label_dir / f"{stem}.txt").write_text(
                "\n".join(label_lines) + ("\n" if label_lines else ""), encoding="utf-8"
            )
            stats["written"] += 1

print(stats)

## 5. 안전장치: 라벨 검사 + split별 개수 확인
지난번 겪었던 "엉뚱한 라벨을 참조하는" 버그가 재발하지 않았는지 전수 검사합니다.

In [ ]:
bad_files = []
for split in SPLIT_RATIOS:
    image_dir = OUT_ROOT / "images" / split
    label_dir = OUT_ROOT / "labels" / split
    image_stems = {p.stem for p in image_dir.iterdir()}
    label_stems = {p.stem for p in label_dir.iterdir()}
    if image_stems != label_stems:
        bad_files.append((split, "이미지/라벨 파일명 불일치"))
    for label_path in label_dir.glob("*.txt"):
        for line in label_path.read_text(encoding="utf-8").splitlines():
            if line.strip() and not line.startswith("0 "):
                bad_files.append((split, str(label_path)))
                break
    print(f"[{split}] 이미지 {len(image_stems)}장, 라벨 {len(label_stems)}개")

if bad_files:
    raise RuntimeError(f"문제 있는 항목 {len(bad_files)}개: {bad_files[:5]}")
print("검사 통과: 모든 split에서 이미지/라벨이 1:1로 맞고 class_id는 전부 0입니다.")

## 6. dataset.yaml

In [ ]:
dataset_yaml_text = (
    f"train: {(OUT_ROOT / 'images' / 'train').resolve()}\n"
    f"val: {(OUT_ROOT / 'images' / 'val').resolve()}\n"
    f"test: {(OUT_ROOT / 'images' / 'test').resolve()}\n"
    "\n"
    "names:\n"
    "  0: tile\n"
)
dataset_yaml_path = OUT_ROOT / "dataset.yaml"
dataset_yaml_path.write_text(dataset_yaml_text, encoding="utf-8")
print(dataset_yaml_text)

## 7. 학습: YOLOv8s (기본으로 신뢰할 모델)

In [ ]:
from ultralytics import YOLO

YOLO_EPOCHS = 100
YOLO_IMGSZ = 960

yolov8s_model = YOLO("yolov8s.pt")
yolov8s_model.train(
    data=str(dataset_yaml_path.resolve()),
    epochs=YOLO_EPOCHS,
    imgsz=YOLO_IMGSZ,
    batch=-1,
    device=0 if torch.cuda.is_available() else "cpu",
    workers=8,
    patience=25,
    project="runs/detect",
    name="rummikub_yolov8s_tile_augmented",
    pretrained=True,
    seed=RANDOM_SEED,
    deterministic=True,
    close_mosaic=10,
    plots=True,
)

yolov8s_best_path = Path(yolov8s_model.trainer.best)
yolov8s_out_path = WEIGHTS_OUT / "yolov8s_tile_detector_best.pt"
shutil.copy2(yolov8s_best_path, yolov8s_out_path)
print("YOLOv8s 가중치 저장:", yolov8s_out_path.resolve())

## 8. 학습: YOLO26n (비교용)
이전에 1클래스로도 전혀 감지를 못 했던 모델이라, 데이터를 늘리고 val/test를 갖춘 지금
상태에서 다시 시도해서 YOLOv8s와 성능을 비교해봅니다.

In [ ]:
yolo26n_model = YOLO("yolo26n.pt")
yolo26n_model.train(
    data=str(dataset_yaml_path.resolve()),
    epochs=YOLO_EPOCHS,
    imgsz=YOLO_IMGSZ,
    batch=-1,
    device=0 if torch.cuda.is_available() else "cpu",
    workers=8,
    patience=25,
    project="runs/detect",
    name="rummikub_yolo26n_tile_augmented",
    pretrained=True,
    seed=RANDOM_SEED,
    deterministic=True,
    close_mosaic=10,
    plots=True,
)

yolo26n_best_path = Path(yolo26n_model.trainer.best)
yolo26n_out_path = WEIGHTS_OUT / "yolo26n_tile_detector_best.pt"
shutil.copy2(yolo26n_best_path, yolo26n_out_path)
print("YOLO26n 가중치 저장:", yolo26n_out_path.resolve())

## 9. test 성능 나란히 비교

In [ ]:
def eval_on_test(model: YOLO, name: str):
    metrics = model.val(
        data=str(dataset_yaml_path.resolve()),
        split="test",
        imgsz=YOLO_IMGSZ,
        device=0 if torch.cuda.is_available() else "cpu",
        project="runs/detect",
        name=f"{name}_eval",
        plots=True,
    )
    print(
        f"{name:12s} P={metrics.box.mp:.3f} R={metrics.box.mr:.3f} "
        f"mAP50={metrics.box.map50:.3f} mAP50-95={metrics.box.map:.3f}"
    )


eval_on_test(yolov8s_model, "yolov8s")
eval_on_test(yolo26n_model, "yolo26n")